# Geographical event characterisation
Descriptive analysis of saved exposure and ecological outputs. No raw inventory scans or spatial calculations. Event prefixes describe geographical events, not necessarily countries (for example, Mekong).

Run cells in order. Set `REGENERATE_ONLY = True` to read only the saved analysis tables and redraw the three figures. A new numerical analysis requires a fresh output directory; existing results are never overwritten.

## 1. Configuration and input paths

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "geography").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "geography").is_dir():
    raise RuntimeError("Open this notebook from the GeoTest-Atlas root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from geography.event_analysis import (
    load_characterisation, aggregate_events, save_analysis, load_analysis)
from geography.plotting import (
    plot_event_exposure, plot_event_proximity, plot_event_ecology)

EXPOSURE_CSV = PROJECT_ROOT / "outputs/exposure/all_chips/exposure_summary.csv"
ENRICHMENT_ROOT = PROJECT_ROOT / "outputs/enrichment"
DOWNSTREAM_PARQUET = ENRICHMENT_ROOT / "enriched/sen1floods11_enriched.parquet"
OUTPUT = PROJECT_ROOT / "outputs/analysis/geographical_events"
REGENERATE_ONLY = False
EVENT_METADATA = None  # Optional CSV with chip_id,event for other downstream datasets.
EXPECTED_CHIPS = 446  # Set explicitly for another dataset, or None.
EXPECTED_EVENTS = 11
EXPECTED_INVENTORIES = ("croma_ssl4eo", "terramind_majortom", "terramind_ssl4eo", "terramind")
RADII = (5, 10, 25, 50, 100)
EVENT_ORDER = None  # None: alphabetical; otherwise supply every event exactly once.
EXPOSURE_LABELS = {"croma_ssl4eo": "SSL4EO-S12 (CROMA)", "terramind": "TerraMesh (TerraMind)"}
ECOLOGICAL_LABELS = {"downstream": "Sen1Floods11", "ssl4eo": "SSL4EO-S12", "terramesh": "TerraMesh"}
TOP_ECOREGIONS = 8
PROXIMITY_LOG1P = True
SELECTED_CHIP = "Ghana_313799"
SELECTED_EVENT = "Ghana"


## 2. Input loading and compatibility validation
The loader checks IDs, complete exposure rows, splits, positive-area flags, ecological hashes, full-inventory totals, RESOLVE versions and recorded source names. Exposure provenance lacks source content hashes; category CSVs also lack a cryptographic link to their producing Parquet. These limits are recorded and warned about, rather than treated as verified historical source equivalence. Only the small downstream Parquet is opened.

In [ ]:
if REGENERATE_ONLY:
    tables = load_analysis(OUTPUT)
    chips = tables["chip_level_characterisation"]
else:
    chips, inventories, provenance = load_characterisation(
        EXPOSURE_CSV, DOWNSTREAM_PARQUET, ENRICHMENT_ROOT,
        event_metadata=EVENT_METADATA, expected_chips=EXPECTED_CHIPS,
        expected_events=EXPECTED_EVENTS, expected_inventories=EXPECTED_INVENTORIES, radii=RADII)
display(chips.head())


## 3. Geographical event identification
The Sen1Floods11 adapter uses the verified event-prefix/numeric-ID naming convention. Official split membership is preserved. For another dataset supply explicit event/domain metadata.

In [ ]:
membership = chips[["chip_id", "event", "split"]].drop_duplicates()
display(pd.crosstab(membership.event, membership.split, margins=True))
event_order = sorted(membership.event.unique()) if EVENT_ORDER is None else list(EVENT_ORDER)
if len(event_order) != membership.event.nunique() or set(event_order) != set(membership.event):
    raise ValueError("EVENT_ORDER must contain every event exactly once")


## 4. Individual chip and event inspection

In [ ]:
display(chips.loc[chips.chip_id.eq(SELECTED_CHIP)])
display(chips.loc[chips.event.eq(SELECTED_EVENT)])
ecological_chips = chips.drop_duplicates("chip_id")
display(pd.crosstab(ecological_chips.event, ecological_chips.assignment_status))
# Use the existing world inventory HTML map for geographical exploration.


## 5. Event-level geographical exposure
Coverage is the arithmetic mean of existing chip union-coverage fractions, not event territorial coverage. Positive-area overlap differs from zero-distance boundary contact. Standard deviations use ddof=1; quartiles use pandas linear interpolation. Counts refer to records or exact-coordinate centres per chip, never globally deduplicated event counts.

In [ ]:
if not REGENERATE_ONLY:
    tables = aggregate_events(chips, inventories, radii=RADII)
display(tables["event_exposure_summary"])


## 6. Multiscale geographical proximity
These counts use the saved Haversine centre-to-centre radii. They are distinct from nearest-footprint distances. Mean and median record counts and distinct-centre counts remain separate.

In [ ]:
display(tables["event_proximity_summary"])


## 7. Biome and ecoregion characterisation
Composition uses all event chips as the denominator. Original assignment statuses are retained, including ambiguous common-biome assignments. Pretraining frequencies use all inventory records, including unresolved records. Residual groups have no available distinct-centre totals and retain missing values. Category absence is zero only in a validated full category table.

In [ ]:
display(tables["event_biome_composition"])
display(tables["event_ecoregion_composition"])
display(tables["ecological_inventory_representation"])


## 8. Numerical summary tables
All event/category relationships are exported. The provenance file is written last as the completion marker. If an interrupted analysis leaves a directory without provenance, use a new output path; no source outputs are removed.

In [ ]:
if not REGENERATE_ONLY:
    save_analysis(tables, provenance, OUTPUT)
print(f"Numerical results: {OUTPUT}")


## 9. Compact scientific figures
Exactly three figures, each PDF and 300 DPI PNG. Existing scientific_style encloses construction, layout and export. RESOLVE colours are retained. The proximity colour scale uses log1p only when configured (zero stays zero); hatched heatmap cells mean missing. Ecoregions are selected by downstream frequency only. Boxplots retain all distances including zeros; whiskers follow Matplotlib's 1.5-IQR convention, with outliers displayed.

In [ ]:
figures = OUTPUT / "figures"
plot_event_exposure(tables, figures, inventories=EXPOSURE_LABELS, event_order=event_order)
plot_event_proximity(tables, figures, inventories=EXPOSURE_LABELS,
                     event_order=event_order, radii=RADII, log1p=PROXIMITY_LOG1P)
plot_event_ecology(tables, figures, inventory_labels=ECOLOGICAL_LABELS,
                   event_order=event_order, top_ecoregions=TOP_ECOREGIONS)
print(f"Three figures exported to {figures}")


## Interpretation limits
Overlap indicates potential geographical exposure, not proven historical observation reuse or downstream label leakage. SSL4EO footprints are approximate. Inventory size and repeated acquisitions affect counts. Ecological assignments describe centres, not complete footprints; ecological membership implies neither proximity nor spectral similarity. Chips within an event may be spatially correlated. These summaries describe sampled chips, not the whole event territory. No temporal/product matching, performance correlation, hypothesis tests or new splits are performed.